# Análisis de Preguntas Derivadas 3 y 4 — ELA-NOM

Este cuaderno operacionaliza y responde dos preguntas metodológicas planteadas por los jurados:

1. **Pregunta Derivada 4 (Tamaño de Muestra y Curva de Aprendizaje):** ¿Cómo evoluciona el error (MAE) del modelo a medida que aumenta el número de contiendas históricas disponibles para entrenamiento?
2. **Pregunta Derivada 3 (Tracción Tardía):** ¿Aporta la métrica de crecimiento de interacciones en los últimos días de campaña información predictiva adicional sobre la cuota final de voto?

In [ ]:
import pandas as pd
import numpy as np
import scipy.special as sp
import scipy.stats as stats
import matplotlib.pyplot as plt

# Cargar dataset
df = pd.read_csv('Herramientas/ipynb antiguo/dataset_modelo_electoral.csv')

# Preprocesamiento
df['total_votos_muni'] = df.groupby('Municipio')['Votos'].transform('sum')
df['cuota_votos_real'] = df['Votos'] / df['total_votos_muni']
df['total_likes_muni'] = df.groupby('Municipio')['total_likes'].transform('sum')
df['dominancia_likes'] = np.where(df['total_likes_muni'] > 0, df['total_likes'] / df['total_likes_muni'], 0)

print(f"Total municipios: {df['Municipio'].nunique()}, Total candidatos: {len(df)}")

## 1. Curva de Aprendizaje (Pregunta Derivada 4)

Evaluación de la reducción del MAE fuera de muestra al variar el número de contiendas en el dataset de entrenamiento ($N \in \{5, 10, 15, 20, 25, 30\}$).

In [ ]:
sample_sizes = [5, 10, 15, 20, 25, 30]
n_repit = 50
learning_results = []
municipios = df['Municipio'].unique()

np.random.seed(42)
for size in sample_sizes:
    maes_size = []
    for _ in range(n_repit):
        train_munis = np.random.choice(municipios, size=size, replace=False)
        test_munis = [m for m in municipios if m not in train_munis]
        if len(test_munis) == 0:
            test_munis = train_munis
        
        train_df = df[df['Municipio'].isin(train_munis)]
        test_df = df[df['Municipio'].isin(test_munis)]
        
        slope, intercept = np.polyfit(train_df['total_likes'], train_df['cuota_votos_real'], 1)
        preds = np.clip(slope * test_df['total_likes'] + intercept, 0, 1)
        mae = np.mean(np.abs(preds - test_df['cuota_votos_real'])) * 100
        maes_size.append(mae)
    
    learning_results.append({'Contiendas_Entrenamiento': size, 'MAE_Medio_pp': np.mean(maes_size), 'Desv_Std': np.std(maes_size)})

df_res = pd.DataFrame(learning_results)
df_res

## 2. Evaluación de Tracción Tardía (Pregunta Derivada 3)

Comparación de la correlación de Spearman entre la dominancia acumulada total (14 días) y la dominancia restringida a la tracción tardía (últimos días de campaña).

In [ ]:
df['total_late_growth'] = df['facebook_likes_late_growth'].fillna(0) + df['tiktok_likes_late_growth'].fillna(0) + df['twitter_likes_late_growth'].fillna(0)
df['total_late_muni'] = df.groupby('Municipio')['total_late_growth'].transform('sum')
df['dominancia_late'] = np.where(df['total_late_muni'] > 0, df['total_late_growth'] / df['total_late_muni'], 0)

rho_total, _ = stats.spearmanr(df['dominancia_likes'], df['cuota_votos_real'])
rho_late, _ = stats.spearmanr(df['dominancia_late'], df['cuota_votos_real'])

print(f"Spearman Dominancia Total (14 días): rho = {rho_total:.4f}")
print(f"Spearman Dominancia Tracción Tardía (últimos días): rho = {rho_late:.4f}")